In [9]:
import pandas as pd
import yaml

In [10]:
def load_rules(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)["attacks"]
    
def convert_timestamps(df):
    # Zeek timestamps are Unix epoch (UTC)
    df["datetime"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    # Convert to CICIDS2017 experiment timezone
    df["datetime"] = df["datetime"].dt.tz_convert("Canada/Atlantic")
    df["time"] = df["datetime"].dt.time
    return df

def label_flows_vectorized(df, rules):
    df["label"] = "BENIGN"

    for rule in rules:
        start = pd.to_datetime(rule["start"]).time()
        end = pd.to_datetime(rule["end"]).time()

        attacker_ips = rule["attacker_ips"]
        victim_ips = rule["victim_ips"]

        mask = (
            (df["time"] >= start) &
            (df["time"] <= end) &
            (
                (
                    df["id.orig_h"].isin(attacker_ips) &
                    df["id.resp_h"].isin(victim_ips)
                )
                |
                (
                    df["id.orig_h"].isin(victim_ips) &
                    df["id.resp_h"].isin(attacker_ips)
                )
            )
        )

        df.loc[mask, "label"] = rule["label"]

        print(rule["name"], "matches:", mask.sum())
    
    return df

In [11]:
def generate_tsv(conn_path, rules_path, dataset_output_name):
    df = pd.read_csv(conn_path, delimiter="\t")
    df = convert_timestamps(df)

    print("Dataset datetime range:")
    print(df["datetime"].min())
    print(df["datetime"].max())

    print("\nDataset time range:")
    print(df["time"].min(), "->", df["time"].max())

    rules = load_rules(rules_path)
    print("Label distribution:")
    df = label_flows_vectorized(df, rules)
    df.drop(columns=["datetime", "time"], inplace=True)
    print(df["label"].value_counts())
    df.to_csv(f"{dataset_output_name}_labeled.tsv", sep='\t', index=False, header=True)

In [12]:
generate_tsv("zeek_logs_wednesday/conn.tsv", "rules/wednesday.yaml", "wednesday")

Dataset datetime range:
2017-07-05 08:42:42.084372044-03:00
2017-07-05 17:10:14.557249069-03:00

Dataset time range:
08:42:42.084372 -> 17:10:14.557249
Label distribution:
DoS_Slowloris matches: 3864
DoS_Slowhttptest matches: 15929
DoS_Hulk matches: 154769
DoS_GoldenEye matches: 7739
Heartbleed matches: 1
label
BENIGN              327060
DOS_HTTP_Flood      154769
DOS_SLOWHTTPTEST     15929
DOS_GOLDENEYE         7739
DOS_SLOWLORIS         3864
HEARTBLEED               1
Name: count, dtype: int64


In [13]:
generate_tsv("zeek_logs_friday/conn.tsv", "rules/friday.yaml", "friday")

Dataset datetime range:
2017-07-07 08:59:50.315195084-03:00
2017-07-07 17:02:40.977797985-03:00

Dataset time range:
08:59:50.315195 -> 17:02:40.977797
Label distribution:
Botnet_ARES matches: 735
PortScan_01 matches: 75
PortScan_02 matches: 73
PortScan_03 matches: 0
PortScan_04 matches: 0
PortScan_05 matches: 0
PortScan_06 matches: 0
PortScan_07 matches: 114
PortScan_08 matches: 37
PortScan_09 matches: 0
PortScan_10 matches: 0
PortScan_11 matches: 0
PortScan_12 matches: 0
PortScan_13 matches: 55113
PortScan_14 matches: 80176
PortScan_15 matches: 0
PortScan_16 matches: 0
PortScan_17 matches: 2
PortScan_18 matches: 0
PortScan_19 matches: 9126
PortScan_20 matches: 156
PortScan_21 matches: 35
PortScan_22 matches: 0
PortScan_23 matches: 0
PortScan_24 matches: 9127
PortScan_25 matches: 0
PortScan_26 matches: 0
PortScan_27 matches: 0
DDoS_LOIT matches: 94614
label
BENIGN       297970
PORTSCAN     154034
DDOS_LOIT     94614
BOTNET          735
Name: count, dtype: int64
